# Automatic generation of Notebook using PyCropML
    This notebook implements a crop model.

### Model Stomataresistance

In [ ]:
MODULE list_sub
    IMPLICIT NONE
    TYPE container
        CLASS(*), ALLOCATABLE :: item
        CLASS(*), ALLOCATABLE :: items(:)
    END TYPE

    interface Add
        module procedure AddToListFloat
        module procedure AddToListInt
        module procedure AddToListChar
        module procedure AddToListIntArray
    end interface
CONTAINS

    FUNCTION indice(vectorElem, elem)
        CHARACTER(LEN=*), DIMENSION(:):: vectorElem
        INTEGER::iterator, indice
        CHARACTER(LEN=*):: elem
        DO iterator= 1, SIZE(vectorElem)
            IF(vectorElem(iterator)==elem) THEN
                indice = iterator
            END IF
        END DO
        RETURN
    END FUNCTION indice


    SUBROUTINE AddToListFloat(list, element)
        IMPLICIT NONE
        INTEGER :: i, isize
        REAL, INTENT(IN) :: element
        REAL, DIMENSION(:), ALLOCATABLE, INTENT(INOUT) :: list
        REAL, DIMENSION(:), ALLOCATABLE :: clist
        IF(ALLOCATED(list)) THEN
            isize = SIZE(list)
            ALLOCATE(clist(isize+1))
            DO i=1,isize
                clist(i) = list(i)
            END DO
            clist(isize+1) = element
            DEALLOCATE(list)
            CALL MOVE_ALLOC(clist, list)
        ELSE
            ALLOCATE(list(1))
            list(1) = element
        END IF
    END SUBROUTINE AddToListFloat

    SUBROUTINE AddToListInt(list, element)
        IMPLICIT NONE
        INTEGER :: i, isize
        INTEGER, INTENT(IN) :: element
        INTEGER, DIMENSION(:), ALLOCATABLE, INTENT(INOUT) :: list
        INTEGER, DIMENSION(:), ALLOCATABLE :: clist
        IF(ALLOCATED(list)) THEN
            isize = SIZE(list)
            ALLOCATE(clist(isize+1))
            DO i=1,isize
                clist(i) = list(i)
            END DO
            clist(isize+1) = element
            DEALLOCATE(list)
            CALL MOVE_ALLOC(clist, list)
        ELSE
            ALLOCATE(list(1))
            list(1) = element
        END IF
    END SUBROUTINE AddToListInt

    SUBROUTINE AddToListChar(list, element)
        IMPLICIT NONE
        INTEGER :: i, isize, l
        CHARACTER(LEN=*), INTENT(IN) :: element
        CHARACTER(LEN=*), DIMENSION(:), ALLOCATABLE, INTENT(INOUT) :: list
        CHARACTER(LEN=65), DIMENSION(:), ALLOCATABLE :: clist
        IF(ALLOCATED(list)) THEN
            isize = SIZE(list)
            ALLOCATE(clist(isize+1))
            DO i=1,isize
                clist(i) = list(i)
            END DO
            clist(isize+1) = element
            DEALLOCATE(list)
            CALL MOVE_ALLOC(clist, list)
        ELSE
            l=1
            ALLOCATE(list(l))
            list(l) = element
        END IF
    END SUBROUTINE AddToListChar


    SUBROUTINE AddToListIntArray(a, e)
        INTEGER, ALLOCATABLE, INTENT(INOUT) :: a(:)
        INTEGER, ALLOCATABLE, INTENT(IN) :: e(:)
        INTEGER, ALLOCATABLE:: tmp(:)

        IF (.NOT.ALLOCATED(a)) THEN
            ALLOCATE(a(1))
            ALLOCATE(a(SIZE(e)), source = e)
        ELSE
            CALL MOVE_ALLOC(a,tmp)
            ALLOCATE(a(SIZE(tmp)+SIZE(e)), source = e)
            a(1:SIZE(tmp)) = tmp
            a(SIZE(tmp)+1:SIZE(a)) = e
        END IF
    END SUBROUTINE AddToListIntArray

END MODULE list_sub
MODULE Stomataresistancemod
    IMPLICIT NONE
CONTAINS

    SUBROUTINE model_stomataresistance(saturation_beta, &
        stomata_conductance_alpha, &
        carboxylation_pathway, &
        atmospheric_co2_concentration, &
        saturation_vapor_pressure_deficit, &
        gross_photosynthesis_reference_mol, &
        stomata_resistance)
        IMPLICIT NONE
        INTEGER:: i_cyml_r
        REAL, INTENT(IN) :: saturation_beta
        REAL, INTENT(IN) :: stomata_conductance_alpha
        INTEGER, INTENT(IN) :: carboxylation_pathway
        REAL, INTENT(IN) :: atmospheric_co2_concentration
        REAL, INTENT(IN) :: saturation_vapor_pressure_deficit
        REAL, INTENT(IN) :: gross_photosynthesis_reference_mol
        REAL, INTENT(OUT) :: stomata_resistance
        !- Name: StomataResistance -Version: 1, -Time step: 1
        !- Description:
    !            * Title: MONICA stomata resistance
    !            * Authors: Claas Nendel (transcription into Crop2ML by Michael Berg-Mohnicke)
    !            * Reference: 
    !        
    !            * Institution: ZALF e.V.
    !            * ExtendedDescription: 
    !        
    !            * ShortDescription: Calculates the MONICA reference evapotranspiration
        !- inputs:
    !            * name: saturation_beta
    !                          ** description : Original: Yu et al. 2001; beta = 3.5
    !                          ** inputtype : parameter
    !                          ** parametercategory : constant
    !                          ** datatype : DOUBLE
    !                          ** max : 
    !                          ** min : 
    !                          ** default : 2.5
    !                          ** unit : 
    !            * name: stomata_conductance_alpha
    !                          ** description : Original: Yu et al. 2001; alpha = 0.06 mol/m2/s
    !                          ** inputtype : parameter
    !                          ** parametercategory : constant
    !                          ** datatype : DOUBLE
    !                          ** max : 
    !                          ** min : 
    !                          ** default : 40
    !                          ** unit : mmol/m2/s
    !            * name: carboxylation_pathway
    !                          ** description : Is it a C3 (=1) or C4 (=2) crop. (0 = no crop)
    !                          ** inputtype : parameter
    !                          ** parametercategory : constant
    !                          ** datatype : INT
    !                          ** max : 2
    !                          ** min : 0
    !                          ** default : 0
    !                          ** unit : dimensionless
    !            * name: atmospheric_co2_concentration
    !                          ** description : CO2 concentration in the atmosphere. Optional if no crop.
    !                          ** inputtype : variable
    !                          ** variablecategory : exogenous
    !                          ** datatype : DOUBLE
    !                          ** max : 
    !                          ** min : 0
    !                          ** default : 430
    !                          ** unit : ppm
    !            * name: saturation_vapor_pressure_deficit
    !                          ** description : saturation_vapor_pressure_deficit
    !                          ** inputtype : variable
    !                          ** variablecategory : auxiliary
    !                          ** datatype : DOUBLE
    !                          ** max : 
    !                          ** min : 0
    !                          ** default : 0
    !                          ** unit : kPa
    !            * name: gross_photosynthesis_reference_mol
    !                          ** description : gross_photosynthesis_reference_mol (-1 = not supplied = no crop)
    !                          ** inputtype : variable
    !                          ** variablecategory : auxiliary
    !                          ** datatype : DOUBLE
    !                          ** max : 
    !                          ** min : 0
    !                          ** default : -1
    !                          ** unit : [mol m-2 s-1] or [cm3 cm-2 s-1]
        !- outputs:
    !            * name: stomata_resistance
    !                          ** description : stomata resistance (FAO default value is 100)
    !                          ** variablecategory : auxiliary
    !                          ** datatype : DOUBLE
    !                          ** max : 10000
    !                          ** min : 0
    !                          ** unit : s/m
        IF(carboxylation_pathway .GT. 0) THEN
            IF(gross_photosynthesis_reference_mol .LE. 0.0) THEN
                stomata_resistance = 999999.9
            ELSE IF ( carboxylation_pathway .EQ. 1) THEN
                stomata_resistance = atmospheric_co2_concentration * (1.0 +  &
                        (saturation_vapor_pressure_deficit / saturation_beta)) /  &
                        (stomata_conductance_alpha * gross_photosynthesis_reference_mol)
            ELSE
                stomata_resistance = atmospheric_co2_concentration * (1.0 +  &
                        (saturation_vapor_pressure_deficit / saturation_beta)) /  &
                        (stomata_conductance_alpha * gross_photosynthesis_reference_mol)
            END IF
        END IF
    END SUBROUTINE model_stomataresistance

END MODULE
PROGRAM test
    USE Stomataresistancemod

END PROGRAM
